# 03. Лучшие майнеры на train/test: TBR и precision (окно с 2013-01-01)

Первая половина прежних `petri_states_eval_*.ipynb` и `run_split.py`: журнал, разбиение 80/20 (seed 42), сети на train, TBR fitness на train/test и ETC precision на test.

Здесь же сети и варианты трасс test сохраняются в `out/<конфигурация>/prepared/`: `04_alignments.ipynb` работает с ними и журнал заново не строит.

Прервать и запустить снова можно: готовые метрики лежат в `align_ckpt/<модель>_tbr.json`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "common"))
import settings as cfg

PART = "2013"                         # "2013" — с окном, "all" — без окна

CONFIGS = cfg.EVAL_CONFIGS               # конфигурации с alignments


In [2]:
import time, warnings
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
import petri_pipeline as pp
import petri_states as ps
import align_parallel as ap
import pm4py

DB = pp.find_db(cfg.DB_CANDIDATES)
categorize = pp.import_categorizer(cfg.CATEGORIES_DIRS)
print("база:", DB, "| pm4py", pm4py.__version__)

tables, _ = pp.drop_full_duplicates(pp.load_tables(DB))
tables, _, _ = pp.drop_invalid_stays(tables)
hist = pp.build_history_table(tables, categorize)
obs_end = pp.observation_end(hist)
print(f"таблица истории: {len(hist):,} строк, конец наблюдения {obs_end}")



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




база: D:\diploma\jupyter lab\петри_пересчет\data\compas.db | pm4py 2.7.23.9


таблица истории: 300,249 строк, конец наблюдения 2016-04-19 07:19:36


In [3]:
rows = []
for config in CONFIGS:
    c = cfg.CONFIGS[config]
    t0 = time.time()
    log, _, _ = ps.build_state_log(hist, obs_end=obs_end, log_start=cfg.PARTS[PART]["log_start"],
                                   short_days=c["short"], long_days=c["long"], end_rule=c["end_rule"])
    for line in ps.check_state_log(log, c["end_rule"]):
        print("OK:", line)
    train, test = pp.split_people(log, cfg.TEST_SHARE, cfg.SEED)
    tr, te = pp.to_pm4py(train), pp.to_pm4py(test)
    print(f"{config}: журнал за {time.time() - t0:.0f} с; train {train.person_id.nunique():,} людей, "
          f"test {test.person_id.nunique():,}, вариантов в test {len(pm4py.get_variants(te)):,}", flush=True)
    models = ps.discover_named(tr, cfg.EVAL_MODELS[config])
    ap.save_prepared(PART, config, models, te)
    ck = cfg.out_dir(PART, config) / "align_ckpt"
    for name, m in models.items():
        t0 = time.time()
        r = ps.evaluate_split_one(name, m, tr, te, do_align=False, ckpt_dir=ck)
        rows.append(dict(config=config, **r))
        print(f"  {name}: {time.time() - t0:.0f} с, fitness test {r['fitness_tbr_test']:.3f}, "
              f"precision test {r['precision_etc_test']:.3f}", flush=True)
    del log, train, test, tr, te, models

OK: время внутри каждой трассы строго возрастает
OK: перед end: {'free': 10590, 'prison': 647, 'jail': 38, 'arrest+jail': 1, 'arrest+charge+jail': 1}
OK: перед end без заключения и не free: 0, из них не в последние 2 дня: 0
OK: free → free подряд: 0


long180_v2: журнал за 100 с; train 9,022 людей, test 2,255, вариантов в test 1,150


2013 long180_v2: сохранено 6 сетей и 1,150 вариантов -> D:\diploma\jupyter lab\петри_пересчет\с_окном_2013\out\long180_v2\prepared


  DFG_p0.05: 0 с, fitness test 0.968, precision test 0.756


  DFG_p0.0: 0 с, fitness test 1.000, precision test 0.438


  HM_dep0.95: 0 с, fitness test 0.981, precision test 0.601


... (49 outputs left out)

replaying log with TBR, completed traces ::   0%|          | 0/3922 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/1143 [00:00<?, ?it/s]

replaying log with TBR, completed traces ::   0%|          | 0/8044 [00:00<?, ?it/s]

  flower: 8 с, fitness test 1.000, precision test 0.358


In [4]:
cols = ["config", "model", "places", "transitions", "silent", "train_traces", "test_traces", "fitness_tbr_train",
        "fitness_tbr_test", "pct_fit_traces_tbr_test", "precision_etc_test", "sec_tbr", "sec_prec"]
pd.DataFrame(rows)[cols].round(3)

,config,model,places,transitions,silent,train_traces,test_traces,fitness_tbr_train,fitness_tbr_test,pct_fit_traces_tbr_test,precision_etc_test,sec_tbr,sec_prec
0,long180_v2,DFG_p0.05,12,27,1,9022,2255,0.967,0.968,72.417,0.756,3.821,4.243
1,long180_v2,DFG_p0.0,12,73,1,9022,2255,1.000,1.000,100.000,0.438,6.379,10.086
2,long180_v2,HM_dep0.95,15,34,24,9022,2255,0.982,0.981,60.931,0.601,6.138,8.636
3,long180_v2,SplitMiner,19,34,24,9022,2255,0.776,0.779,6.563,0.772,12.549,4.284
4,long180_v2,IM_noise0.2,35,44,34,9022,2255,1.000,1.000,99.867,0.368,8.885,40.667
5,long180_v2,flower,3,12,2,9022,2255,1.000,1.000,100.000,0.353,3.333,5.943
6,long180_v1,DFG_p0.05,12,27,1,9022,2255,0.967,0.967,72.417,0.756,3.907,4.819
7,long180_v1,DFG_p0.0,12,72,1,9022,2255,1.000,1.000,100.000,0.442,6.243,9.700
8,long180_v1,HM_dep0.95,15,34,24,9022,2255,0.982,0.981,60.931,0.601,6.083,9.543
9,long180_v1,SplitMiner,22,34,24,9022,2255,0.789,0.793,6.253,0.777,12.550,3.944
